Silver Layer
- Cleaning bronze tables 
- Simulating SCD type 2 

In [12]:
from pyspark.sql.functions import col, upper, trim, coalesce, lit
from pyspark.sql.functions import to_date, current_date, when
from delta.tables import DeltaTable
from pyspark.sql.types import BooleanType, IntegerType

StatementMeta(, 83196dbb-dc20-43a2-a293-8e8832c8f00b, 14, Finished, Available, Finished, False)

In [13]:
#Loading and cleaning Bronze product data
df_product_raw = spark.read.table('bronze_product') 

df_product_clean = ( 
    df_product_raw .select('ProductID','Name','ProductNumber','ProductSubcategoryID','ListPrice','StandardCost','Color','Size','Weight') 
    .withColumn('Name', upper(trim(col('Name')))) 
    .withColumn('ProductNumber', upper(trim(col('ProductNumber')))) 
    .withColumn('ListPrice', col('ListPrice').cast('double')) 
    .withColumn('StandardCost', col('StandardCost').cast('double')) 
    .withColumn('Weight', col('Weight').cast('double')) 
    .withColumn('ListPrice', coalesce(col('ListPrice'), lit(0.0))) 
    .withColumn('StandardCost', coalesce(col('StandardCost'), lit(0.0))) 
    .filter(col('ProductID').isNotNull()) 
) 

StatementMeta(, 83196dbb-dc20-43a2-a293-8e8832c8f00b, 15, Finished, Available, Finished, False)

In [14]:
#Adding SCD type 2 columns for initial load
df_initial = ( 
    df_product_clean 
    .withColumn('effective_date',lit('2011-01-01').cast('date')) 
    .withColumn('expiry_date',lit('9999-12-31').cast('date')) 
    .withColumn('is_current',lit(True).cast(BooleanType())) 
    .withColumn('record_version',lit(1).cast(IntegerType())) 
) 

StatementMeta(, 83196dbb-dc20-43a2-a293-8e8832c8f00b, 16, Finished, Available, Finished, False)

In [15]:
#Writing initial Silver dimension 
df_initial.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('silver_dim_product') 

print(f'Initial load: {df_initial.count()} products')

StatementMeta(, 83196dbb-dc20-43a2-a293-8e8832c8f00b, 17, Finished, Available, Finished, False)

Initial load: 504 products


Simulating price change in 15 records for SCD Type 2

In [16]:
#15 products get a $100 price increase
#saving this changes in a view
df_updates = ( 
    df_product_clean.limit(15) 
    .withColumn('ListPrice',col('ListPrice') + 100) 
    .withColumn('effective_date',lit('2026-02-01').cast('date')) 
    .withColumn('expiry_date',lit('9999-12-31').cast('date')) 
    .withColumn('is_current',lit(True).cast(BooleanType())) 
    .withColumn('record_version',lit(2).cast(IntegerType())) 
) 

df_updates.createOrReplaceTempView('updates')

StatementMeta(, 83196dbb-dc20-43a2-a293-8e8832c8f00b, 18, Finished, Available, Finished, False)

In [17]:
silver = DeltaTable.forName(spark,'silver_dim_product') 
silver.alias("target").merge(
    df_updates.alias("src"),
    "target.ProductID = src.ProductID AND target.is_current = true"
).whenMatchedUpdate(
    condition="target.ListPrice <> src.ListPrice",
    set={
        "is_current": "false",
        "expiry_date": "current_date()"
    }
).execute()

StatementMeta(, 83196dbb-dc20-43a2-a293-8e8832c8f00b, 19, Finished, Available, Finished, False)

In [18]:
df_updates.write.mode("append").saveAsTable("silver_dim_product")

StatementMeta(, 83196dbb-dc20-43a2-a293-8e8832c8f00b, 20, Finished, Available, Finished, False)

In [19]:
#Verifying history
spark.sql(''' 
    SELECT ProductID, Name, ListPrice, effective_date, expiry_date, is_current 
    FROM  silver_dim_product
    WHERE ProductID IN (SELECT ProductID FROM updates) 
    ORDER BY ProductID, effective_date 
''').show(30, truncate=False) 

StatementMeta(, 83196dbb-dc20-43a2-a293-8e8832c8f00b, 21, Finished, Available, Finished, False)

+---------+---------------------+---------+--------------+-----------+----------+
|ProductID|Name                 |ListPrice|effective_date|expiry_date|is_current|
+---------+---------------------+---------+--------------+-----------+----------+
|1        |ADJUSTABLE RACE      |0.0      |2011-01-01    |2026-03-30 |false     |
|1        |ADJUSTABLE RACE      |100.0    |2026-02-01    |9999-12-31 |true      |
|2        |BEARING BALL         |0.0      |2011-01-01    |2026-03-30 |false     |
|2        |BEARING BALL         |100.0    |2026-02-01    |9999-12-31 |true      |
|3        |BB BALL BEARING      |0.0      |2011-01-01    |2026-03-30 |false     |
|3        |BB BALL BEARING      |100.0    |2026-02-01    |9999-12-31 |true      |
|4        |HEADSET BALL BEARINGS|0.0      |2011-01-01    |2026-03-30 |false     |
|4        |HEADSET BALL BEARINGS|100.0    |2026-02-01    |9999-12-31 |true      |
|316      |BLADE                |0.0      |2011-01-01    |2026-03-30 |false     |
|316      |BLADE

Cleaning remaining tables

In [20]:
#Silver Sales Territory 
df_territory = ( 
    spark.read.table('bronze_sales_territory') 
    .select('TerritoryID','Name','CountryRegionCode','Group') 
    .withColumn('Name',upper(trim(col('Name')))) 
    .withColumn('CountryRegionCode',upper(trim(col('CountryRegionCode')))) 
    .withColumn('Group',upper(trim(col('Group')))) 
    .dropDuplicates(['TerritoryID']) 
) 
df_territory.write.format('delta').mode('overwrite').saveAsTable('silver_dim_territory') 

#Silver Customer 
df_customer = ( 
    spark.read.table('bronze_customer') 
    .select('CustomerID','PersonID','StoreID','TerritoryID') 
    .dropDuplicates(['CustomerID']) 
    .filter(col('CustomerID').isNotNull()) 
) 
df_customer.write.format('delta').mode('overwrite').saveAsTable('silver_dim_customer') 

#Silver Sales Order Detail 
df_detail = ( 
    spark.read.table('bronze_sales_order_detail') 
    .select('SalesOrderID','SalesOrderDetailID','ProductID', 
            'OrderQty','UnitPrice','LineTotal') 
    .withColumn('UnitPrice',col('UnitPrice').cast('double')) 
    .withColumn('LineTotal',col('LineTotal').cast('double')) 
    .withColumn('OrderQty',col('OrderQty').cast('int')) 
    .filter(col('LineTotal') > 0) 
) 
df_detail.write.format('delta').mode('overwrite').saveAsTable('silver_sales_order_detail') 

#Silver Sales Order Header
df_header = ( 
    spark.read.table('bronze_sales_order_header') 
    .select('SalesOrderID','OrderDate','CustomerID','TerritoryID', 
            'SubTotal','TaxAmt','Freight','TotalDue','Status') 
    .withColumn('OrderDate',to_date(col('OrderDate'))) 
    .withColumn('SubTotal',col('SubTotal').cast('double')) 
    .withColumn('TotalDue',col('TotalDue').cast('double')) 
    .filter(col('SalesOrderID').isNotNull()) 
    .dropDuplicates(['SalesOrderID']) 
) 
df_header.write.format('delta').mode('overwrite').saveAsTable('silver_sales_order_header') 

print('All Silver tables created')

StatementMeta(, 83196dbb-dc20-43a2-a293-8e8832c8f00b, 22, Finished, Available, Finished, False)

All Silver tables created
